# Tomato Leaf Disease Dataset Preprocessing (Google Colab Version)

Este notebook foi desenvolvido para rodar no Google Colab e processar um dataset de imagens de folhas de tomate em 10 classes.

### Requisitos implementados:
1. **OS-Agnostic Path Handling**: Uso de `pathlib` para total compatibilidade.
2. **Dataset Balanceado**: Limita cada classe de origem a um máximo configurável (default: 1.000 imagens), totalizando 10.000 imagens no subset 100%.
3. **Escala Logarítmica de Subsets**: Geração de subsets de [1%, 2%, 5%, 10%, 20%, 50%, 100%].
4. **Regra de Inclusão Cumulativa**: O subset de tamanho $P_i$ contém todas as imagens do subset anterior $P_{i-1}$, mantendo as imagens consistentes.
5. **Split Exato de 85% Treino / 15% Teste**: Proporção rigorosa mantida em cada subset.
6. **Rastreamento via Metadata**: Criação de um `dataset_metadata.json` detalhado com a divisão de cada arquivo por subset e split.

--- 
## 1. Imports & Google Drive Mount

Execute esta célula para importar as bibliotecas e montar o seu Google Drive no Colab.

In [ ]:
import json
import random
import shutil
from pathlib import Path
from typing import Dict, List, Tuple

# Montagem do Google Drive para uso no Google Colab
try:
    from google.colab import drive
    print("Montando o Google Drive...")
    drive.mount('/content/drive')
    print("Drive montado com sucesso.")
except ImportError:
    print("Ambiente Colab não detectado. Executando localmente.")

--- 
## 2. Funções (Imports / Processamento)

Esta célula contém a lógica modular do pipeline de dados, incluindo a leitura do dataset, balanceamento de classes, cálculo dos splits cumulativos e cópia dos arquivos.

In [ ]:
def scan_dataset(src_path: Path) -> Dict[str, List[Path]]:
    """
    Escaneia a pasta original buscando classes de imagens.
    """
    if not src_path.exists():
        raise FileNotFoundError(f"Diretório de origem '{src_path}' não encontrado.")

    image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    dataset_structure = {}

    for class_dir in src_path.iterdir():
        if class_dir.is_dir() and not class_dir.name.startswith("."):
            images = [
                f for f in class_dir.iterdir()
                if f.is_file() and f.suffix.lower() in image_extensions
            ]
            # Ordenação garantida antes do shuffle (evita diferenças de SO)
            images.sort()
            if images:
                dataset_structure[class_dir.name] = images

    return dataset_structure


def generate_subsets(
    dataset_structure: Dict[str, List[Path]],
    dest_path: Path,
    seed: int,
    train_ratio: float,
    percentages: List[int],
    src_path: Path,
    max_images_per_class: int,
) -> Dict:
    """
    Gera os subsets cumulativos salvando-os em disco e retorna o dicionário de metadados.
    """
    # Define seed aleatório
    random.seed(seed)

    # Pre-shuffla e limita cada classe para balancear o dataset original
    balanced_dataset = {}
    for class_name, images in dataset_structure.items():
        shuffled = list(images)
        random.shuffle(shuffled)
        balanced_dataset[class_name] = shuffled[:max_images_per_class]
    
    metadata = {
        "original_dataset": {
            "path": str(src_path.resolve().as_posix()),
            "total_images": sum(len(imgs) for imgs in dataset_structure.values()),
            "classes": {c: len(imgs) for c, imgs in dataset_structure.items()}
        },
        "balanced_base": {
            "max_images_per_class": max_images_per_class,
            "total_images": sum(len(imgs) for imgs in balanced_dataset.values()),
            "classes": {c: len(imgs) for c, imgs in balanced_dataset.items()}
        },
        "configuration": {
            "seed": seed,
            "train_ratio": train_ratio,
            "test_ratio": round(1.0 - train_ratio, 2),
            "percentages": percentages
        },
        "subsets": {}
    }

    # Cria pools fixos de Treino e Teste para garantir a consistência das imagens (Inclusão Cumulativa)
    pools = {}
    for class_name, images in balanced_dataset.items():
        n_total = len(images)
        n_train = int(round(n_total * train_ratio))
        
        train_pool = images[:n_train]
        test_pool = images[n_train:]
        pools[class_name] = (train_pool, test_pool)
    # Loop pelos subsets percentuais solicitados
    for p in percentages:
        subset_name = f"subset_{p}%"
        print(f"\nProcessando {subset_name}...")
        
        subset_meta = {
            "percentage": float(p),
            "summary": {
                "total_train": 0,
                "total_test": 0,
                "total_images": 0,
                "classes": {}
            },
            "files": {
                "train": {},
                "test": {}
            }
        }
        
        for class_name, (train_pool, test_pool) in pools.items():
            n_class_total = len(balanced_dataset[class_name])
            
            # Total de imagens da classe no subset atual
            target_class_total = int(round(n_class_total * (p / 100.0)))
            
            # Split de treino e teste proporcional
            target_train = int(round(target_class_total * train_ratio))
            target_test = target_class_total - target_train
            
            # Filtra os primeiros elementos dos pools
            subset_train = train_pool[:target_train]
            subset_test = test_pool[:target_test]
            
            # Pastas de destino
            train_dest_dir = dest_path / subset_name / "train" / class_name
            test_dest_dir = dest_path / subset_name / "test" / class_name
            
            # Garante a existência física das pastas
            train_dest_dir.mkdir(parents=True, exist_ok=True)
            test_dest_dir.mkdir(parents=True, exist_ok=True)
            
            # Copia imagens de Treino e mapeia caminhos
            train_rel_paths = []
            for img_path in subset_train:
                shutil.copy2(img_path, train_dest_dir / img_path.name)
                train_rel_paths.append(f"{class_name}/{img_path.name}")
                
            # Copia imagens de Teste e mapeia caminhos
            test_rel_paths = []
            for img_path in subset_test:
                shutil.copy2(img_path, test_dest_dir / img_path.name)
                test_rel_paths.append(f"{class_name}/{img_path.name}")
                
            # Atualiza resumo da classe
            subset_meta["summary"]["classes"][class_name] = {
                "train": len(subset_train),
                "test": len(subset_test),
                "total": len(subset_train) + len(subset_test)
            }
            
            # Atualiza arquivos mapeados
            subset_meta["files"]["train"][class_name] = train_rel_paths
            subset_meta["files"]["test"][class_name] = test_rel_paths
            
            # Acumula totais do subset
            subset_meta["summary"]["total_train"] += len(subset_train)
            subset_meta["summary"]["total_test"] += len(subset_test)
            subset_meta["summary"]["total_images"] += (len(subset_train) + len(subset_test))
            
        metadata["subsets"][subset_name] = subset_meta
        print(f"  -> {subset_name} finalizado! Treino: {subset_meta['summary']['total_train']} | Teste: {subset_meta['summary']['total_test']} | Total: {subset_meta['summary']['total_images']}")
        
    return metadata

--- 
## 3. Execução Interativa

Configure os caminhos de origem (`SRC_DIR`) e destino (`DEST_DIR`), o número máximo de imagens por classe para balanceamento (`MAX_IMAGES_PER_CLASS`) e rode o processamento.

In [ ]:
# --- CONFIGURAÇÕES INTERATIVAS ---
# Exemplo no Google Drive:
# SRC_DIR = Path("/content/drive/MyDrive/redes_neurais/dataset/orginal/dataset-plantas-com-augmentation")
# DEST_DIR = Path("/content/drive/MyDrive/redes_neurais/dataset/preprocessed")

# Default local para execução de teste
SRC_DIR = Path("../dataset/orginal/dataset-plantas-com-augmentation")
DEST_DIR = Path("../dataset/preprocessed")

SEED = 42
TRAIN_RATIO = 0.85
MAX_IMAGES_PER_CLASS = 1000
PERCENTAGES = [1, 2, 5, 10, 20, 50, 100]

print("=== Pipeline de Preprocessamento de Imagens ===")
print(f"Origem: {SRC_DIR.resolve()}")
print(f"Destino: {DEST_DIR.resolve()}")
print(f"Limite de imagens por classe: {MAX_IMAGES_PER_CLASS}")

try:
    # 1. Escaneia dataset
    print("\n1. Escaneando classes de origem...")
    dataset_structure = scan_dataset(SRC_DIR)
    print(f"Classes encontradas: {len(dataset_structure)}")
    for class_name, files in dataset_structure.items():
        print(f"  - {class_name}: {len(files)} imagens (será balanceada para {min(len(files), MAX_IMAGES_PER_CLASS)})")

    # 2. Limpa subsets antigos no destino para evitar arquivos órfãos
    print("\n2. Limpando diretórios antigos...")
    for p in PERCENTAGES:
        folder = DEST_DIR / f"subset_{p}%"
        if folder.exists():
            print(f"  Removendo: {folder.name}")
            shutil.rmtree(folder)

    # 3. Executa o split e cópia cumulativa
    print("\n3. Executando processamento cumulativo...")
    metadata = generate_subsets(
        dataset_structure=dataset_structure,
        dest_path=DEST_DIR,
        seed=SEED,
        train_ratio=TRAIN_RATIO,
        percentages=PERCENTAGES,
        src_path=SRC_DIR,
        max_images_per_class=MAX_IMAGES_PER_CLASS
    )

    # 4. Salva arquivo JSON de metadados
    metadata_file = DEST_DIR / "dataset_metadata.json"
    with open(metadata_file, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=4, ensure_ascii=False)
    print(f"\n[SUCESSO] Preprocessamento concluído! Metadados em: {metadata_file.resolve()}")

except Exception as e:
    print(f"\n[ERRO] O processamento falhou: {e}")